# 3.4 Combined model confidence and pyDock score

This notebook provides a guided analysis of AlphaFold2/ColabFold and AlphaFold3 local or Server predictions together with their pyDock energies. The default worked example is `4POU`, but the same workflow can be applied to another case by changing `CASE_ID` in the first visible code cell.

**Z_AF/pyDock-1VDW = Z_AF-MC - Z_pyDock-1VDW**

Higher AF-MC is favourable, whereas lower pyDock-1VDW energy is favourable. The combined score therefore highlights models supported by both AlphaFold confidence and pyDock energy without requiring an experimental reference.

> **Quick start**
> 1. Select the **Alphafold_pyDock analysis** kernel.
> 2. Review the first visible code cell and change `CASE_ID` or the model-selection options if needed.
> 3. Use **Run All** from the notebook toolbar.
> 4. Implementation cells are marked as hidden in compatible Jupyter frontends. If VS Code opens them expanded, collapse the cell input from the left cell bar; the outputs remain visible.

### Notebook guide

[Configuration](#1-configuration-and-repository-paths) · [Input inventory](#2-input-inventory) · [Combined ranking](#9-confidence-normalization-and-combined-ranking) · [Top-N table, 3D viewer and export](#10-explore-and-export-the-top-ranked-models) · [Plots](#11-interpret-the-main-plots) · [Evidence and interpretation template](#13-evidence-summary-and-interpretation-template) · [Generated files](#14-generated-files-and-scope)

When reference-based pyDock results are available, the tables also include ligand RMSD, interface RMSD, FNC, DockQ and the quality class. These retrospective annotations help assess the `4POU` worked example but do not change the primary ranking.


## 1. Configuration and repository paths

This is the only code cell that users normally need to edit. It remains visible so that the selected case and analysis options can be checked before running the notebook.

The default cohort follows the protocol recommendation: relaxed AF2 recycle models and available AF3 local or Server models as predicted. Unrelaxed AF2 models can be enabled only for a technical test before relaxation and pyDock rescoring are complete.

Set REPO_ROOT_OVERRIDE to an absolute path, or define the AF_PYDOCK_REPO_ROOT environment variable, when the current working directory is outside this repository.


In [ ]:
import configparser
import hashlib
import json
import os
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
import pandas as pd
import py3Dmol
from IPython.display import clear_output, display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

CASE_ID = "4POU"

INCLUDE_AF2_UNRELAXED = False
INCLUDE_AF2_RELAXED = True
INCLUDE_AF3_AS_PREDICTED = True
DROP_DUPLICATE_AF2_FINALS = True
GENERATE_PLOTS = True
DISPLAY_PLOTS = True
DEFAULT_TOP_N = 10
MAX_TOP_N = 50

# Example manual override:
# REPO_ROOT_OVERRIDE = Path("/absolute/path/to/AF_pyDock")
REPO_ROOT_OVERRIDE = None


In [ ]:
REPO_MARKERS = (
    "4_Case_Studies",
    "3.4_Combined_Model_Confidence_and_pyDock_Score",
)

def find_repo_root(override=None):
    '''Find the repository from an override, an environment variable, or cwd.'''
    starts = []
    if override is not None:
        starts.append(Path(override).expanduser())
    env_root = os.environ.get("AF_PYDOCK_REPO_ROOT")
    if env_root:
        starts.append(Path(env_root).expanduser())
    starts.append(Path.cwd())

    checked = []
    for start in starts:
        start = start.resolve()
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.append(candidate)
            if all((candidate / marker).exists() for marker in REPO_MARKERS):
                return candidate

    raise FileNotFoundError(
        "Repository root not found. Set REPO_ROOT_OVERRIDE or "
        "AF_PYDOCK_REPO_ROOT to the AF_pyDock repository."
    )

REPO_ROOT = find_repo_root(REPO_ROOT_OVERRIDE)
CASE_DIR = REPO_ROOT / "4_Case_Studies" / CASE_ID
AF2_DIR = CASE_DIR / "AF2"
AF3_DIR = CASE_DIR / "AF3"
AF3_RESULTS_DIR = AF3_DIR / "output" / CASE_ID
AF3_SERVER_DIRS = (
    sorted(path for path in AF3_DIR.glob("fold*") if path.is_dir())
    if AF3_DIR.is_dir()
    else []
)
ANALYSIS_DIR = CASE_DIR / "Analysis"

if not CASE_DIR.is_dir():
    raise FileNotFoundError(f"Case directory not found: {CASE_DIR}")
if INCLUDE_AF2_UNRELAXED or INCLUDE_AF2_RELAXED:
    if not AF2_DIR.is_dir():
        print(f"Warning: AF2 directory not found: {AF2_DIR}")
if (
    INCLUDE_AF3_AS_PREDICTED
    and not AF3_RESULTS_DIR.is_dir()
    and not AF3_SERVER_DIRS
):
    print(f"Warning: no local or Server AF3 results found below: {AF3_DIR}")

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root : {REPO_ROOT}")
print(f"Case directory  : {CASE_DIR}")
print(f"Analysis output : {ANALYSIS_DIR}")


## 2. Input inventory

The notebook looks inside the selected case for model PDB files, pyDock `.ene` tables, AF2 logs, AF3 confidence JSON files and, when available, `.RMSD.capri` reference results. Only complete model structures are candidates: the auxiliary receptor and ligand PDB files produced by pyDock are excluded.

The inventory below provides an early check that the files prepared in sections 3.1 and 3.3.2 are available before the ranking is calculated.


In [ ]:
AF2_ENERGY_FILES = sorted(AF2_DIR.rglob("*.ene")) if AF2_DIR.is_dir() else []
AF3_LOCAL_ENERGY_FILES = (
    sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*.ene"))
    if AF3_RESULTS_DIR.is_dir()
    else []
)
AF3_SERVER_ENERGY_FILES = sorted(
    energy_path
    for server_dir in AF3_SERVER_DIRS
    for energy_path in server_dir.glob("*.ene")
)
AF3_ENERGY_FILES = sorted(
    [*AF3_LOCAL_ENERGY_FILES, *AF3_SERVER_ENERGY_FILES]
)
AF2_CAPRI_FILES = (
    sorted(AF2_DIR.rglob("*.RMSD.capri"))
    if AF2_DIR.is_dir()
    else []
)
AF3_LOCAL_CAPRI_FILES = (
    sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*.RMSD.capri"))
    if AF3_RESULTS_DIR.is_dir()
    else []
)
AF3_SERVER_CAPRI_FILES = sorted(
    capri_path
    for server_dir in AF3_SERVER_DIRS
    for capri_path in server_dir.glob("*.RMSD.capri")
)
AF3_CAPRI_FILES = sorted(
    [*AF3_LOCAL_CAPRI_FILES, *AF3_SERVER_CAPRI_FILES]
)
AF2_LOG_FILES = sorted(AF2_DIR.glob("*/log.txt")) if AF2_DIR.is_dir() else []
AF3_LOCAL_CONFIDENCE_FILES = (
    sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*_summary_confidences.json"))
    if AF3_RESULTS_DIR.is_dir()
    else []
)
AF3_SERVER_CONFIDENCE_FILES = sorted(
    confidence_path
    for server_dir in AF3_SERVER_DIRS
    for confidence_path in server_dir.glob("*_summary_confidences_[0-9]*.json")
)
AF3_CONFIDENCE_FILES = sorted(
    [*AF3_LOCAL_CONFIDENCE_FILES, *AF3_SERVER_CONFIDENCE_FILES]
)

raw_inventory = pd.DataFrame(
    [
        ("AF2 energy files found", len(AF2_ENERGY_FILES)),
        ("Local AF3 energy files found", len(AF3_LOCAL_ENERGY_FILES)),
        ("AF3 Server energy files found", len(AF3_SERVER_ENERGY_FILES)),
        ("AF2 CAPRI files found", len(AF2_CAPRI_FILES)),
        ("Local AF3 CAPRI files found", len(AF3_LOCAL_CAPRI_FILES)),
        ("AF3 Server CAPRI files found", len(AF3_SERVER_CAPRI_FILES)),
        ("AF2 logs found", len(AF2_LOG_FILES)),
        ("Local AF3 confidence JSON found", len(AF3_LOCAL_CONFIDENCE_FILES)),
        ("AF3 Server confidence JSON found", len(AF3_SERVER_CONFIDENCE_FILES)),
    ],
    columns=["item", "count"],
)
print(raw_inventory.to_string(index=False))


## 3. Helper functions and identifiers

The stable identifier is model_id, derived from the verified primary PDB name. AF2, local AF3 and AF3 Server filenames are parsed separately because they encode different metadata.


In [ ]:
AF2_MODEL_RE = re.compile(
    rf"^(?P<case>{re.escape(CASE_ID)})_"
    r"(?P<state>unrelaxed|relaxed)_rank_(?P<rank>\d{3})_"
    r"(?P<af_version>alphafold2_multimer_v\d+)_"
    r"model_(?P<model_number>\d+)_seed_(?P<seed>\d+)"
    r"(?:\.r(?P<recycle>\d+))?$",
    re.IGNORECASE,
)
AF3_LOCAL_MODEL_RE = re.compile(
    rf"^(?P<case>{re.escape(CASE_ID)})_seed-(?P<seed>\d+)_"
    r"sample-(?P<sample>\d+)_model$",
    re.IGNORECASE,
)
AF3_SERVER_MODEL_RE = re.compile(
    rf"^fold_(?P<case>{re.escape(CASE_ID)})_model_(?P<model_index>\d+)$",
    re.IGNORECASE,
)

def path_relative_to_repo(path):
    path = Path(path).resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()

def file_digest(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def files_are_identical(first, second):
    first, second = Path(first), Path(second)
    return (
        first.is_file()
        and second.is_file()
        and first.stat().st_size == second.stat().st_size
        and file_digest(first) == file_digest(second)
    )

def is_auxiliary_pdb(path):
    stem = Path(path).stem.lower()
    return stem.endswith("_rec") or stem.endswith("_lig")

def parse_model_metadata(model_path):
    model_path = Path(model_path)
    af2_match = AF2_MODEL_RE.fullmatch(model_path.stem)
    if af2_match:
        groups = af2_match.groupdict()
        return {
            "case_id": groups["case"].upper(),
            "model_id": model_path.stem,
            "prediction_method": "AF2",
            "af_version": groups["af_version"].lower(),
            "relaxation_status": groups["state"].lower(),
            "af2_filename_rank": int(groups["rank"]),
            "model_number": int(groups["model_number"]),
            "seed": int(groups["seed"]),
            "recycle": (
                int(groups["recycle"])
                if groups["recycle"] is not None
                else None
            ),
            "sample": None,
        }

    af3_local_match = AF3_LOCAL_MODEL_RE.fullmatch(model_path.stem)
    if af3_local_match:
        groups = af3_local_match.groupdict()
        return {
            "case_id": groups["case"].upper(),
            "model_id": model_path.stem,
            "prediction_method": "AF3",
            "af_version": "alphafold3",
            "relaxation_status": "as_predicted",
            "af2_filename_rank": None,
            "model_number": None,
            "seed": int(groups["seed"]),
            "recycle": None,
            "sample": int(groups["sample"]),
        }

    af3_server_match = AF3_SERVER_MODEL_RE.fullmatch(model_path.stem)
    if af3_server_match:
        groups = af3_server_match.groupdict()
        return {
            "case_id": groups["case"].upper(),
            "model_id": model_path.stem,
            "prediction_method": "AF3",
            "af_version": "alphafold3",
            "relaxation_status": "as_predicted",
            "af2_filename_rank": None,
            "model_number": None,
            "seed": None,
            "recycle": None,
            "sample": None,
        }

    return None

def cohort_is_enabled(metadata):
    if metadata["prediction_method"] == "AF2":
        if metadata["relaxation_status"] == "unrelaxed":
            return INCLUDE_AF2_UNRELAXED
        if metadata["relaxation_status"] == "relaxed":
            return INCLUDE_AF2_RELAXED
        return False
    if metadata["prediction_method"] == "AF3":
        return INCLUDE_AF3_AS_PREDICTED
    return False


In [ ]:
def model_path_from_ini(energy_path):
    '''Resolve the original PDB from the pyDock ini file when possible.'''
    energy_path = Path(energy_path)
    ini_path = energy_path.with_suffix(".ini")
    if not ini_path.is_file():
        return None

    parser = configparser.ConfigParser(interpolation=None)
    parser.read(ini_path)
    for section in ("receptor", "ligand"):
        if not parser.has_option(section, "pdb"):
            continue
        value = parser.get(section, "pdb").strip().strip("\"'")
        candidate = Path(value).expanduser()
        if not candidate.is_absolute():
            candidate = ini_path.parent / candidate
        candidate = candidate.resolve()
        if candidate.is_file() and not is_auxiliary_pdb(candidate):
            return candidate
    return None

def resolve_model_path(energy_path):
    '''Use the companion ini first, then remove an explicit ligand suffix.'''
    energy_path = Path(energy_path)
    from_ini = model_path_from_ini(energy_path)
    if from_ini is not None:
        return from_ini, "ini"

    model_stem = re.sub(
        r"_LIG_[^.]+$",
        "",
        energy_path.stem,
        flags=re.IGNORECASE,
    )
    fallback = energy_path.with_name(f"{model_stem}.pdb").resolve()
    if fallback.is_file() and not is_auxiliary_pdb(fallback):
        return fallback, "validated_suffix"
    return None, None

def recycle20_energy_path(final_energy_path):
    final_energy_path = Path(final_energy_path)
    match = re.fullmatch(
        r"(?P<model>.+?)(?P<ligand>_LIG_[^.]+)",
        final_energy_path.stem,
        flags=re.IGNORECASE,
    )
    if not match:
        return None
    filename = (
        f"{match.group('model')}.r20{match.group('ligand')}"
        f"{final_energy_path.suffix}"
    )
    return final_energy_path.with_name(filename)

def duplicate_af2_final(model_path, energy_path=None):
    '''Return true only when an unsuffixed AF2 final duplicates recycle 20.'''
    model_path = Path(model_path)
    metadata = parse_model_metadata(model_path)
    if (
        metadata is None
        or metadata["prediction_method"] != "AF2"
        or metadata["recycle"] is not None
    ):
        return False, None

    recycle_model = model_path.with_name(f"{model_path.stem}.r20.pdb")
    if not files_are_identical(model_path, recycle_model):
        return False, None

    if energy_path is not None:
        recycle_energy = recycle20_energy_path(energy_path)
        if recycle_energy is None or not files_are_identical(
            energy_path, recycle_energy
        ):
            return False, None

    return True, recycle_model


In [ ]:
ENE_ENERGY_COLUMNS = ["Ele", "Desolv", "VDW", "Total"]
RMSD_NUMERIC_COLUMNS = ["l_RMSD", "i_RMSD", "FNC", "DockQ"]
CAPRI_STAR_CLASSES = {
    "": "Incorrect",
    "*": "Acceptable",
    "**": "Medium",
    "***": "High",
}
CAPRI_TEXT_CLASSES = {
    "incorrect": "Incorrect",
    "acceptable": "Acceptable",
    "medium": "Medium",
    "high": "High",
}

def numeric_or_nan(value):
    value = pd.to_numeric(value, errors="coerce")
    return float(value) if pd.notna(value) else np.nan

def read_energy_file(energy_path):
    table = pd.read_csv(energy_path, sep=r"\s+", skiprows=[1])
    missing = [
        column for column in ENE_ENERGY_COLUMNS if column not in table
    ]
    if missing:
        raise ValueError(f"Missing pyDock columns {missing}")
    if table.empty:
        raise ValueError("Empty pyDock energy table")

    if len(table) > 1 and "RANK" in table:
        rank_values = pd.to_numeric(table["RANK"], errors="coerce")
        row = table.loc[rank_values.sort_values().index[0]]
    else:
        row = table.iloc[0]

    values = {
        column: numeric_or_nan(row[column])
        for column in ("Ele", "Desolv", "VDW")
    }
    values["pyDock-0.1VDW"] = numeric_or_nan(row["Total"])

    selected_conf = numeric_or_nan(row.get("Conf", np.nan))
    ene_l_rmsd = np.nan
    for column in ("l_RMSD", "RMSD"):
        if column not in row.index:
            continue
        candidate = numeric_or_nan(row[column])
        if pd.notna(candidate):
            ene_l_rmsd = candidate
            break

    return values, selected_conf, ene_l_rmsd

def capri_path_for_energy(energy_path):
    energy_path = Path(energy_path)
    return energy_path.with_name(f"{energy_path.stem}.RMSD.capri")

def read_capri_row(capri_path, selected_conf):
    lines = Path(capri_path).read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    header_index = None
    columns = None
    for index, line in enumerate(lines):
        tokens = line.split()
        if (
            tokens
            and tokens[0] == "Conf"
            and {"l_RMSD", "i_RMSD", "FNC"}.issubset(tokens)
        ):
            header_index = index
            columns = tokens
            break
    if header_index is None:
        raise ValueError("CAPRI header not found")
    if len(columns) != len(set(columns)):
        raise ValueError("Duplicate CAPRI column names")

    rows = []
    for line in lines[header_index + 1:]:
        stripped = line.strip()
        if not stripped or set(stripped) <= {"-"}:
            continue
        fields = stripped.split()
        if columns[-1] == "CAPRI" and len(fields) == len(columns) - 1:
            fields.append("")
        if len(fields) != len(columns):
            raise ValueError(
                f"Unexpected CAPRI row with {len(fields)} fields"
            )
        rows.append(dict(zip(columns, fields)))
    if not rows:
        raise ValueError("Empty CAPRI result table")

    if pd.notna(selected_conf):
        matching = [
            row
            for row in rows
            if pd.notna(numeric_or_nan(row.get("Conf")))
            and np.isclose(
                numeric_or_nan(row.get("Conf")),
                selected_conf,
            )
        ]
        if len(matching) != 1:
            raise ValueError(
                f"Expected one CAPRI row for Conf {selected_conf:g}, "
                f"found {len(matching)}"
            )
        return matching[0], columns

    if len(rows) != 1:
        raise ValueError(
            "Cannot select a CAPRI row without a unique Conf value"
        )
    return rows[0], columns

def normalize_capri_class(value, has_dockq):
    value = "" if value is None else str(value).strip()
    if value == "" and has_dockq:
        return pd.NA
    if value in CAPRI_STAR_CLASSES:
        return CAPRI_STAR_CLASSES[value]
    return CAPRI_TEXT_CLASSES.get(value.lower(), value)

def read_capri_metrics(capri_path, selected_conf):
    row, columns = read_capri_row(capri_path, selected_conf)
    metrics = {
        column: numeric_or_nan(row.get(column, np.nan))
        for column in ("l_RMSD", "i_RMSD", "FNC")
    }
    missing = [
        column for column, value in metrics.items() if pd.isna(value)
    ]
    if missing:
        raise ValueError(f"Missing CAPRI numeric values {missing}")

    has_dockq = "DockQ" in columns
    metrics["DockQ"] = (
        numeric_or_nan(row.get("DockQ", np.nan))
        if has_dockq
        else np.nan
    )
    metrics["CAPRI"] = normalize_capri_class(
        row.get("CAPRI"), has_dockq
    )
    metrics["rmsd_source"] = "capri"
    return metrics

def reference_metrics_for_energy(energy_path, selected_conf, ene_l_rmsd):
    capri_path = capri_path_for_energy(energy_path)
    capri_error = None
    if capri_path.is_file():
        try:
            return read_capri_metrics(capri_path, selected_conf), None
        except Exception as exc:
            capri_error = {
                "path": path_relative_to_repo(capri_path),
                "detail": str(exc),
            }

    if pd.notna(ene_l_rmsd):
        return (
            {
                "l_RMSD": float(ene_l_rmsd),
                "i_RMSD": np.nan,
                "FNC": np.nan,
                "DockQ": np.nan,
                "CAPRI": pd.NA,
                "rmsd_source": "ene_fallback",
            },
            capri_error,
        )

    return (
        {
            "l_RMSD": np.nan,
            "i_RMSD": np.nan,
            "FNC": np.nan,
            "DockQ": np.nan,
            "CAPRI": pd.NA,
            "rmsd_source": "missing",
        },
        capri_error,
    )

def load_pydock_energies(energy_paths):
    diagnostics = {
        "associated_pdb": [],
        "missing_pdb": [],
        "unrecognized_model": [],
        "excluded_duplicate": [],
        "excluded_by_flags": [],
        "read_errors": [],
        "capri_read_errors": [],
    }
    records = []

    for energy_path in sorted(map(Path, energy_paths)):
        model_path, resolution = resolve_model_path(energy_path)
        if model_path is None:
            diagnostics["missing_pdb"].append(
                {"path": path_relative_to_repo(energy_path), "detail": ""}
            )
            continue

        diagnostics["associated_pdb"].append(
            {"path": path_relative_to_repo(energy_path), "detail": resolution}
        )
        metadata = parse_model_metadata(model_path)
        if metadata is None:
            diagnostics["unrecognized_model"].append(
                {
                    "path": path_relative_to_repo(energy_path),
                    "detail": path_relative_to_repo(model_path),
                }
            )
            continue

        if not cohort_is_enabled(metadata):
            diagnostics["excluded_by_flags"].append(
                {
                    "path": path_relative_to_repo(energy_path),
                    "detail": metadata["relaxation_status"],
                }
            )
            continue

        if (
            DROP_DUPLICATE_AF2_FINALS
            and metadata["prediction_method"] == "AF2"
            and metadata["recycle"] is None
        ):
            is_duplicate, recycle_model = duplicate_af2_final(
                model_path, energy_path
            )
            if is_duplicate:
                diagnostics["excluded_duplicate"].append(
                    {
                        "path": path_relative_to_repo(energy_path),
                        "detail": path_relative_to_repo(recycle_model),
                    }
                )
                continue

        try:
            energy_values, selected_conf, ene_l_rmsd = read_energy_file(
                energy_path
            )
        except Exception as exc:
            diagnostics["read_errors"].append(
                {
                    "path": path_relative_to_repo(energy_path),
                    "detail": str(exc),
                }
            )
            continue

        rmsd_values, capri_error = reference_metrics_for_energy(
            energy_path,
            selected_conf,
            ene_l_rmsd,
        )
        if capri_error is not None:
            diagnostics["capri_read_errors"].append(capri_error)

        records.append(
            {
                **metadata,
                "model_path": path_relative_to_repo(model_path),
                "energy_path": path_relative_to_repo(energy_path),
                "pdb_resolution": resolution,
                **energy_values,
                **rmsd_values,
            }
        )

    frame = pd.DataFrame(records)
    if not frame.empty:
        for column in RMSD_NUMERIC_COLUMNS:
            frame[column] = pd.to_numeric(
                frame[column], errors="coerce"
            ).astype("Float64")
        frame["CAPRI"] = frame["CAPRI"].astype("string")
        frame["rmsd_source"] = frame["rmsd_source"].astype("string")

        key = ["case_id", "prediction_method", "model_id"]
        duplicated = frame.duplicated(key, keep=False)
        if duplicated.any():
            examples = frame.loc[duplicated, key + ["energy_path"]]
            raise ValueError(
                "More than one pyDock energy file maps to a model:\n"
                + examples.to_string(index=False)
            )
    return frame, diagnostics


In [ ]:
def inventory_primary_models():
    records = []
    diagnostics = {
        "excluded_duplicate": [],
        "excluded_by_flags": [],
        "ignored_auxiliary": [],
    }

    af2_pdbs = sorted(AF2_DIR.rglob("*.pdb")) if AF2_DIR.is_dir() else []
    for model_path in af2_pdbs:
        if is_auxiliary_pdb(model_path):
            diagnostics["ignored_auxiliary"].append(
                {"path": path_relative_to_repo(model_path), "detail": ""}
            )
            continue
        metadata = parse_model_metadata(model_path)
        if metadata is None:
            continue
        if not cohort_is_enabled(metadata):
            diagnostics["excluded_by_flags"].append(
                {
                    "path": path_relative_to_repo(model_path),
                    "detail": metadata["relaxation_status"],
                }
            )
            continue

        if DROP_DUPLICATE_AF2_FINALS and metadata["recycle"] is None:
            energy_candidates = sorted(
                model_path.parent.glob(f"{model_path.stem}_LIG_*.ene")
            )
            if energy_candidates:
                duplicate, recycle_model = duplicate_af2_final(
                    model_path, energy_candidates[0]
                )
            else:
                duplicate, recycle_model = duplicate_af2_final(model_path)
            if duplicate:
                diagnostics["excluded_duplicate"].append(
                    {
                        "path": path_relative_to_repo(model_path),
                        "detail": path_relative_to_repo(recycle_model),
                    }
                )
                continue

        records.append(
            {**metadata, "model_path": path_relative_to_repo(model_path)}
        )

    af3_local_pdbs = (
        sorted(AF3_RESULTS_DIR.glob("seed-*_sample-*/*_model.pdb"))
        if AF3_RESULTS_DIR.is_dir()
        else []
    )
    af3_server_pdbs = sorted(
        model_path
        for server_dir in AF3_SERVER_DIRS
        for model_path in server_dir.glob("*_model_*.pdb")
    )
    af3_pdbs = sorted([*af3_local_pdbs, *af3_server_pdbs])
    for model_path in af3_pdbs:
        metadata = parse_model_metadata(model_path)
        if metadata is None:
            continue
        if not cohort_is_enabled(metadata):
            diagnostics["excluded_by_flags"].append(
                {
                    "path": path_relative_to_repo(model_path),
                    "detail": metadata["relaxation_status"],
                }
            )
            continue
        records.append(
            {**metadata, "model_path": path_relative_to_repo(model_path)}
        )

    frame = pd.DataFrame(records)
    if not frame.empty:
        key = ["case_id", "prediction_method", "model_id"]
        if frame.duplicated(key).any():
            raise ValueError("Duplicate primary model identifiers were found")
    return frame, diagnostics


## 4. Confidence readers

AF2 confidence is read from recycle entries in each ColabFold log. The notebook never overrides the structural state encoded in a PDB filename: relaxed models remain labelled relaxed, and unrelaxed models remain labelled unrelaxed.


In [ ]:
FLOAT_PATTERN = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"
AF2_LOG_RE = re.compile(
    rf"(?P<af_version>alphafold2_multimer_v\d+)_"
    rf"model_(?P<model_number>\d+)_seed_(?P<seed>\d+)\s+"
    rf"recycle=(?P<recycle>\d+)\s+"
    rf"pLDDT=(?P<plddt>{FLOAT_PATTERN})\s+"
    rf"pTM=(?P<ptm>{FLOAT_PATTERN})\s+"
    rf"ipTM=(?P<iptm>{FLOAT_PATTERN})"
    rf"(?:\s+tol=(?P<tol>{FLOAT_PATTERN}))?"
)

def load_af2_confidence(log_paths):
    records = []
    for log_path in sorted(map(Path, log_paths)):
        with log_path.open(encoding="utf-8", errors="replace") as handle:
            for line in handle:
                match = AF2_LOG_RE.search(line)
                if not match:
                    continue
                groups = match.groupdict()
                records.append(
                    {
                        "case_id": CASE_ID,
                        "prediction_method": "AF2",
                        "af_version": groups["af_version"].lower(),
                        "model_number": int(groups["model_number"]),
                        "seed": int(groups["seed"]),
                        "recycle": int(groups["recycle"]),
                        "pLDDT": float(groups["plddt"]),
                        "pTM": float(groups["ptm"]),
                        "ipTM": float(groups["iptm"]),
                        "ranking_score": np.nan,
                        "tol": (
                            float(groups["tol"])
                            if groups["tol"] is not None
                            else np.nan
                        ),
                        "confidence_path": path_relative_to_repo(log_path),
                    }
                )

    frame = pd.DataFrame(records)
    if not frame.empty:
        keys = [
            "case_id",
            "prediction_method",
            "af_version",
            "model_number",
            "seed",
            "recycle",
        ]
        duplicated = frame.duplicated(keys, keep=False)
        if duplicated.any():
            raise ValueError(
                "Duplicate AF2 confidence records:\n"
                + frame.loc[duplicated, keys].to_string(index=False)
            )
    return frame


Local AF3 confidence is read from summary JSON files inside seed/sample directories, while AF3 Server confidence is read from the indexed summary JSON beside each Server model. The local root aggregate summary is deliberately ignored. pLDDT is left unavailable because it is not present in these summary files.


In [ ]:
AF3_LOCAL_SUMMARY_RE = re.compile(
    rf"^(?P<case>{re.escape(CASE_ID)})_seed-(?P<seed>\d+)_"
    r"sample-(?P<sample>\d+)_summary_confidences$",
    re.IGNORECASE,
)
AF3_SERVER_SUMMARY_RE = re.compile(
    rf"^fold_(?P<case>{re.escape(CASE_ID)})_"
    r"summary_confidences_(?P<model_index>\d+)$",
    re.IGNORECASE,
)

def json_for_csv(value):
    if value is None:
        return None
    return json.dumps(value, separators=(",", ":"))

def load_af3_confidence(summary_paths):
    records = []
    issues = []

    for summary_path in sorted(map(Path, summary_paths)):
        local_match = AF3_LOCAL_SUMMARY_RE.fullmatch(summary_path.stem)
        server_match = AF3_SERVER_SUMMARY_RE.fullmatch(summary_path.stem)
        if not local_match and not server_match:
            issues.append(
                {
                    "path": path_relative_to_repo(summary_path),
                    "detail": "unrecognized summary filename",
                }
            )
            continue

        try:
            payload = json.loads(summary_path.read_text(encoding="utf-8"))
        except Exception as exc:
            issues.append(
                {
                    "path": path_relative_to_repo(summary_path),
                    "detail": str(exc),
                }
            )
            continue

        if local_match:
            groups = local_match.groupdict()
            seed = int(groups["seed"])
            sample = int(groups["sample"])
            model_id = (
                f"{groups['case']}_seed-{seed}_sample-{sample}_model"
            )
        else:
            groups = server_match.groupdict()
            seed = None
            sample = None
            model_id = (
                f"fold_{groups['case']}_model_{groups['model_index']}"
            )

        chain_ids = list(dict.fromkeys(payload.get("chain_ids") or []))
        chain_ptm = payload.get("chain_ptm")
        chain_pair_iptm = payload.get("chain_pair_iptm")

        chain_a_ptm = np.nan
        chain_b_ptm = np.nan
        iptm_a_vs_b = np.nan
        if isinstance(chain_ptm, list):
            if len(chain_ptm) > 0:
                chain_a_ptm = chain_ptm[0]
            if len(chain_ptm) > 1:
                chain_b_ptm = chain_ptm[1]
        if (
            isinstance(chain_pair_iptm, list)
            and len(chain_pair_iptm) > 0
            and isinstance(chain_pair_iptm[0], list)
            and len(chain_pair_iptm[0]) > 1
        ):
            iptm_a_vs_b = chain_pair_iptm[0][1]

        records.append(
            {
                "case_id": groups["case"].upper(),
                "model_id": model_id,
                "prediction_method": "AF3",
                "af_version": "alphafold3",
                "seed": seed,
                "sample": sample,
                "pLDDT": np.nan,
                "pTM": pd.to_numeric(payload.get("ptm"), errors="coerce"),
                "ipTM": pd.to_numeric(payload.get("iptm"), errors="coerce"),
                "ranking_score": pd.to_numeric(
                    payload.get("ranking_score"), errors="coerce"
                ),
                "fraction_disordered": pd.to_numeric(
                    payload.get("fraction_disordered"), errors="coerce"
                ),
                "has_clash": payload.get("has_clash"),
                "num_recycles": payload.get("num_recycles"),
                "chain_ids": json_for_csv(chain_ids),
                "chain_ptm": json_for_csv(chain_ptm),
                "chain_pair_iptm": json_for_csv(chain_pair_iptm),
                "chain_A_pTM": chain_a_ptm,
                "chain_B_pTM": chain_b_ptm,
                "ipTM_A_vs_B": iptm_a_vs_b,
                "confidence_path": path_relative_to_repo(summary_path),
            }
        )

    frame = pd.DataFrame(records)
    if not frame.empty:
        keys = ["case_id", "prediction_method", "model_id"]
        duplicated = frame.duplicated(keys, keep=False)
        if duplicated.any():
            raise ValueError(
                "Duplicate AF3 confidence records:\n"
                + frame.loc[duplicated, keys].to_string(index=False)
            )
    return frame, issues


In [ ]:
def safe_zscore(series):
    '''Return population Z-scores; a constant valid cohort receives zero.'''
    numeric = pd.to_numeric(series, errors="coerce")
    result = pd.Series(np.nan, index=series.index, dtype=float)
    valid = numeric.dropna()
    if valid.empty:
        return result

    standard_deviation = valid.std(ddof=0)
    if not np.isfinite(standard_deviation) or np.isclose(
        standard_deviation, 0.0
    ):
        result.loc[valid.index] = 0.0
    else:
        result.loc[valid.index] = (
            valid - valid.mean()
        ) / standard_deviation
    return result


## 5. Load pyDock energies, reference metrics and the selected model cohort

The native Total field is read from each pyDock energy file and exposed in the table as pyDock-0.1VDW; the source `.ene` remains unchanged. This score corresponds to Ele + Desolv + 0.1 × VDW. The full van der Waals score, pyDock-1VDW, is calculated explicitly as Ele + Desolv + VDW. Both metrics are retained and ranked across all selected models in each case.

For reference-based assessment, a valid matching `.RMSD.capri` file has priority and supplies `l_RMSD`, `i_RMSD`, `FNC`, optional `DockQ` and the normalized quality class. If that file is absent or unreadable, an `RMSD` or `l_RMSD` value in the `.ene` file is retained as `l_RMSD` with `rmsd_source=ene_fallback`; the other reference metrics remain missing. RMSD availability is not required for the confidence–energy ranking.


In [ ]:
model_inventory, inventory_diagnostics = inventory_primary_models()
energy_df, energy_diagnostics = load_pydock_energies(
    [*AF2_ENERGY_FILES, *AF3_ENERGY_FILES]
)

has_relaxed_af2_energy = (
    not energy_df.empty
    and energy_df["prediction_method"].eq("AF2").any()
    and energy_df.loc[
        energy_df["prediction_method"].eq("AF2"),
        "relaxation_status",
    ].eq("relaxed").any()
)
if INCLUDE_AF2_RELAXED and not has_relaxed_af2_energy:
    raise ValueError(
        "The recommended workflow requires relaxed AF2 models with "
        "pyDock .ene files. Run AF2 relaxation and section 3.3.2 first. "
        "For a technical notebook test only, set "
        "INCLUDE_AF2_RELAXED=False and INCLUDE_AF2_UNRELAXED=True."
    )

print(f"Primary models selected : {len(model_inventory)}")
print(f"Models with pyDock data : {len(energy_df)}")
if not energy_df.empty:
    print(
        energy_df.groupby(
            ["prediction_method", "relaxation_status"]
        ).size().to_string()
    )


## 6. Load AF2 confidence


In [ ]:
af2_confidence = load_af2_confidence(AF2_LOG_FILES)
print(f"AF2 confidence records: {len(af2_confidence)}")
if not af2_confidence.empty:
    print(
        af2_confidence.groupby("af_version").size().to_string()
    )


In [ ]:
af2_confidence

## 7. Load AF3 confidence


In [ ]:
af3_confidence, af3_confidence_issues = load_af3_confidence(
    AF3_CONFIDENCE_FILES
)
print(f"AF3 confidence records: {len(af3_confidence)}")
if af3_confidence_issues:
    print(pd.DataFrame(af3_confidence_issues).to_string(index=False))


## 8. Build the consolidated analysis table

Each model is paired with its confidence and pyDock results. Only models with a verified PDB, a pyDock energy table and the corresponding AlphaFold confidence record continue to the consolidated table. The reported merge counts make missing or unmatched inputs visible.


In [ ]:
af2_energy = energy_df.loc[
    energy_df["prediction_method"].eq("AF2")
].copy()
af3_energy = energy_df.loc[
    energy_df["prediction_method"].eq("AF3")
].copy()

AF2_MERGE_KEYS = [
    "case_id",
    "prediction_method",
    "af_version",
    "model_number",
    "seed",
    "recycle",
]
AF3_MERGE_KEYS = [
    "case_id",
    "prediction_method",
    "af_version",
    "model_id",
]

if af2_energy.empty and af2_confidence.empty:
    af2_merged = pd.DataFrame()
    af2_merge_counts = {}
else:
    af2_merged = af2_energy.merge(
        af2_confidence,
        how="outer",
        on=AF2_MERGE_KEYS,
        validate="many_to_one",
        indicator="merge_status",
    )
    af2_merge_counts = af2_merged["merge_status"].value_counts().to_dict()

if af3_energy.empty and af3_confidence.empty:
    af3_merged = pd.DataFrame()
    af3_merge_counts = {}
else:
    af3_merged = af3_energy.merge(
        af3_confidence,
        how="outer",
        on=AF3_MERGE_KEYS,
        validate="one_to_one",
        indicator="merge_status",
    )
    af3_merge_counts = af3_merged["merge_status"].value_counts().to_dict()

matched_frames = []
if not af2_merged.empty:
    matched_frames.append(
        af2_merged.loc[af2_merged["merge_status"].eq("both")]
        .drop(columns="merge_status")
        .copy()
    )
if not af3_merged.empty:
    matched_frames.append(
        af3_merged.loc[af3_merged["merge_status"].eq("both")]
        .drop(columns="merge_status")
        .copy()
    )

if not matched_frames:
    raise ValueError("No model has both pyDock energy and confidence data")

consolidated = pd.concat(matched_frames, ignore_index=True, sort=False)
model_key = ["case_id", "prediction_method", "model_id"]
if consolidated.duplicated(model_key).any():
    raise ValueError("The consolidated model identifier is not unique")

print(f"AF2 merge status: {af2_merge_counts}")
print(f"AF3 merge status: {af3_merge_counts}")
print(f"Models successfully merged: {len(consolidated)}")


## 9. Confidence, normalization and combined ranking

AF-MC is 0.8 × ipTM + 0.2 × pTM. The notebook calculates independent per-case rankings and Z-scores for pyDock-0.1VDW and pyDock-1VDW. The main combined score follows the chapter nomenclature:

**Z_AF/pyDock-1VDW = Z_AF-MC - Z_pyDock-1VDW**

All normalizations and energy rankings are calculated across the selected cohort for each case. Missing reference metrics remain missing and do not exclude a model from this ranking. A zero-variance ranked metric receives a Z-score of zero rather than causing an error.


In [ ]:
consolidated["AF-MC"] = (
    0.8 * pd.to_numeric(consolidated["ipTM"], errors="coerce")
    + 0.2 * pd.to_numeric(consolidated["pTM"], errors="coerce")
)
consolidated["pyDock-1VDW"] = (
    pd.to_numeric(consolidated["Ele"], errors="coerce")
    + pd.to_numeric(consolidated["Desolv"], errors="coerce")
    + pd.to_numeric(consolidated["VDW"], errors="coerce")
)

consolidated["rank_pyDock-0.1VDW"] = (
    consolidated.groupby("case_id")["pyDock-0.1VDW"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)
consolidated["rank_pyDock-1VDW"] = (
    consolidated.groupby("case_id")["pyDock-1VDW"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)

consolidated["Z_AF-MC"] = consolidated.groupby(
    "case_id", group_keys=False
)["AF-MC"].transform(safe_zscore)
consolidated["Z_pyDock-0.1VDW"] = consolidated.groupby(
    "case_id", group_keys=False
)["pyDock-0.1VDW"].transform(safe_zscore)
consolidated["Z_pyDock-1VDW"] = consolidated.groupby(
    "case_id", group_keys=False
)["pyDock-1VDW"].transform(safe_zscore)
consolidated["Z_AF/pyDock-1VDW"] = (
    consolidated["Z_AF-MC"]
    - consolidated["Z_pyDock-1VDW"]
)

required_for_ranking = [
    "model_path",
    "energy_path",
    "confidence_path",
    "AF-MC",
    "pyDock-0.1VDW",
    "pyDock-1VDW",
    "rank_pyDock-0.1VDW",
    "rank_pyDock-1VDW",
    "Z_AF-MC",
    "Z_pyDock-0.1VDW",
    "Z_pyDock-1VDW",
    "Z_AF/pyDock-1VDW",
]
rankable = consolidated[required_for_ranking].notna().all(axis=1)

ranking = consolidated.loc[rankable].copy()
ranking = ranking.sort_values(
    [
        "Z_AF/pyDock-1VDW",
        "AF-MC",
        "pyDock-1VDW",
        "pyDock-0.1VDW",
        "model_id",
    ],
    ascending=[False, False, True, True, True],
    kind="mergesort",
).reset_index(drop=True)
ranking["rank_Z_AF/pyDock-1VDW"] = pd.Series(
    np.arange(1, len(ranking) + 1), dtype="Int64"
)

rank_lookup = ranking[
    model_key + ["rank_Z_AF/pyDock-1VDW"]
]
consolidated = consolidated.merge(
    rank_lookup,
    how="left",
    on=model_key,
    validate="one_to_one",
)
consolidated["rank_Z_AF/pyDock-1VDW"] = consolidated[
    "rank_Z_AF/pyDock-1VDW"
].astype("Int64")

COLUMN_ORDER = [
    "case_id",
    "model_id",
    "prediction_method",
    "af_version",
    "relaxation_status",
    "model_path",
    "energy_path",
    "confidence_path",
    "Ele",
    "Desolv",
    "VDW",
    "pyDock-0.1VDW",
    "pyDock-1VDW",
    "l_RMSD",
    "i_RMSD",
    "FNC",
    "DockQ",
    "CAPRI",
    "rmsd_source",
    "rank_pyDock-0.1VDW",
    "rank_pyDock-1VDW",
    "pLDDT",
    "pTM",
    "ipTM",
    "ranking_score",
    "AF-MC",
    "Z_AF-MC",
    "Z_pyDock-0.1VDW",
    "Z_pyDock-1VDW",
    "Z_AF/pyDock-1VDW",
    "rank_Z_AF/pyDock-1VDW",
]
EXTRA_COLUMNS = [
    "af2_filename_rank",
    "model_number",
    "seed",
    "recycle",
    "sample",
    "tol",
    "fraction_disordered",
    "has_clash",
    "num_recycles",
    "chain_ids",
    "chain_ptm",
    "chain_pair_iptm",
    "chain_A_pTM",
    "chain_B_pTM",
    "ipTM_A_vs_B",
    "pdb_resolution",
]
export_columns = [
    column
    for column in [*COLUMN_ORDER, *EXTRA_COLUMNS]
    if column in consolidated.columns
]

consolidated_export = consolidated[export_columns].sort_values(
    ["prediction_method", "af_version", "model_id"],
    kind="mergesort",
)
ranking_export = ranking[
    [column for column in export_columns if column in ranking.columns]
].sort_values("rank_Z_AF/pyDock-1VDW", kind="mergesort")

INTEGER_EXPORT_COLUMNS = [
    "rank_pyDock-0.1VDW",
    "rank_pyDock-1VDW",
    "rank_Z_AF/pyDock-1VDW",
    "af2_filename_rank",
    "model_number",
    "seed",
    "recycle",
    "sample",
    "num_recycles",
]
for export_frame in (consolidated_export, ranking_export):
    for column in INTEGER_EXPORT_COLUMNS:
        if column in export_frame.columns:
            export_frame[column] = pd.to_numeric(
                export_frame[column], errors="coerce"
            ).astype("Int64")

consolidated_path = ANALYSIS_DIR / "consolidated_results.csv"
ranking_path = ANALYSIS_DIR / "combined_ranking.csv"
consolidated_export.to_csv(consolidated_path, index=False)
ranking_export.to_csv(ranking_path, index=False)

print(f"Consolidated table: {consolidated_path}")
print(f"Combined ranking  : {ranking_path}")
print(f"Ranked models     : {len(ranking_export)}")


## 10. Explore and export the top-ranked models

The static table records the default combined ranking. The interactive panel then lets the user compare ranking criteria, filter AF2 or AF3 models, choose the number of candidates, inspect a model in 3D and save a self-contained copy of the current selection.

Changing these controls only reorders the models already calculated for the manually configured `CASE_ID`; it does not recalculate scores or modify the main CSV files. Reference-based columns such as RMSD and CAPRI are annotations and are not used to select the primary combined ranking.


In [ ]:
ZSCORE_SUMMARY_COLUMNS = [
    "case_id",
    "model_id",
    "prediction_method",
    "af_version",
    "relaxation_status",
    "Z_AF/pyDock-1VDW",
    "rank_Z_AF/pyDock-1VDW",
    "Z_AF-MC",
    "Z_pyDock-1VDW",
    "Z_pyDock-0.1VDW",
    "AF-MC",
    "pyDock-1VDW",
    "pyDock-0.1VDW",
    "l_RMSD",
    "i_RMSD",
    "FNC",
    "DockQ",
    "CAPRI",
    "rmsd_source",
    "rank_pyDock-1VDW",
    "rank_pyDock-0.1VDW",
    "model_path",
]
zscore_summary = ranking_export[
    [column for column in ZSCORE_SUMMARY_COLUMNS if column in ranking_export]
].copy()
zscore_summary["method_group"] = zscore_summary["prediction_method"]
if "af_version" in zscore_summary:
    af2_version = zscore_summary["af_version"].astype("string").str.extract(
        r"_v(\d+)$", expand=False
    )
    af2_with_version = (
        zscore_summary["method_group"].eq("AF2")
        & af2_version.notna()
    )
    zscore_summary.loc[af2_with_version, "prediction_method"] = (
        "AF2 v" + af2_version.loc[af2_with_version]
    )

TOP_TABLE_COLUMNS = [
    "selection_rank",
    "model_id",
    "prediction_method",
    "relaxation_status",
    "Z_AF/pyDock-1VDW",
    "rank_Z_AF/pyDock-1VDW",
    "Z_AF-MC",
    "Z_pyDock-1VDW",
    "Z_pyDock-0.1VDW",
    "AF-MC",
    "pyDock-1VDW",
    "rank_pyDock-1VDW",
    "l_RMSD",
    "i_RMSD",
    "FNC",
    "DockQ",
    "CAPRI",
    "rmsd_source",
]
ZSCORE_FORMATTERS = {
    "Z_AF/pyDock-1VDW": "{:.3f}",
    "Z_AF-MC": "{:.3f}",
    "Z_pyDock-1VDW": "{:.3f}",
    "Z_pyDock-0.1VDW": "{:.3f}",
    "AF-MC": "{:.3f}",
    "pyDock-1VDW": "{:.3f}",
    "l_RMSD": "{:.3f}",
    "i_RMSD": "{:.3f}",
    "FNC": "{:.3f}",
    "DockQ": "{:.3f}",
}

def style_zscore_table(frame):
    formatters = {
        column: formatter
        for column, formatter in ZSCORE_FORMATTERS.items()
        if column in frame.columns
    }
    return frame.style.format(formatters, na_rep="").hide(axis="index")

if DEFAULT_TOP_N < 1 or MAX_TOP_N < 1 or DEFAULT_TOP_N > MAX_TOP_N:
    raise ValueError(
        "Require 1 <= DEFAULT_TOP_N <= MAX_TOP_N for the Top-N controls"
    )

static_top_n = min(DEFAULT_TOP_N, len(zscore_summary))
if static_top_n:
    print(f"Default Top {static_top_n}: Z_AF/pyDock-1VDW")
    static_top = zscore_summary.head(static_top_n).copy()
    static_top.insert(
        0, "selection_rank", np.arange(1, static_top_n + 1)
    )
    display(
        style_zscore_table(
            static_top[TOP_TABLE_COLUMNS]
        )
    )
else:
    print("No rankable models are available for the Top-N table.")


In [ ]:
def close_previous_interactive_panel():
    previous_observers = globals().get(
        "_analysis_interactive_observers", []
    )
    for widget, callback, names in previous_observers:
        try:
            widget.unobserve(callback, names=names)
        except Exception:
            pass

    previous_button = globals().get("_analysis_interactive_button")
    if previous_button is not None:
        button, callback = previous_button
        try:
            button.on_click(callback, remove=True)
        except Exception:
            pass

    previous_widgets = globals().get(
        "_analysis_interactive_widgets", []
    )
    for widget in reversed(previous_widgets):
        try:
            widget.close()
        except Exception:
            pass

    globals()["_analysis_interactive_observers"] = []
    globals()["_analysis_interactive_button"] = None
    globals()["_analysis_interactive_widgets"] = []

close_previous_interactive_panel()
clear_output(wait=False)

RANKING_CRITERIA = {
    "Combined Z (higher is better)": ("Z_AF/pyDock-1VDW", False),
    "AlphaFold confidence Z (higher is better)": ("Z_AF-MC", False),
    "pyDock-1VDW Z (lower is better)": ("Z_pyDock-1VDW", True),
    "pyDock-0.1VDW Z (lower is better)": ("Z_pyDock-0.1VDW", True),
}
TOP_SORT_TIE_BREAKERS = [
    ("Z_AF/pyDock-1VDW", False),
    ("Z_AF-MC", False),
    ("Z_pyDock-1VDW", True),
    ("Z_pyDock-0.1VDW", True),
    ("model_id", True),
]

criterion_widget = widgets.Dropdown(
    options=list(RANKING_CRITERIA),
    value=next(iter(RANKING_CRITERIA)),
    description="Ranking criterion:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)
method_options = [
    "All",
    *sorted(zscore_summary["method_group"].dropna().unique()),
]
method_widget = widgets.Dropdown(
    options=method_options,
    value="All",
    description="Method:",
    style={"description_width": "initial"},
)
top_n_limit = max(1, min(MAX_TOP_N, len(zscore_summary)))
top_n_widget = widgets.IntSlider(
    value=max(1, min(DEFAULT_TOP_N, top_n_limit)),
    min=1,
    max=top_n_limit,
    step=1,
    description="Top N:",
    continuous_update=False,
    disabled=zscore_summary.empty,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)
model_widget = widgets.Dropdown(
    options=[],
    description="3D model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%"),
)
table_output = widgets.Output()
viewer_output = widgets.Output()
save_selection_output = widgets.Output()
model_row_lookup = {}
SELECTIONS_DIR = ANALYSIS_DIR / "selections"

save_selection_button = widgets.Button(
    description="Save current Top N",
    icon="save",
    button_style="success",
    tooltip=(
        "Copy the displayed Top-N table and PDB models into "
        "Analysis/selections"
    ),
    disabled=zscore_summary.empty,
    layout=widgets.Layout(width="210px"),
)

def selected_top_models():
    subset = zscore_summary.copy()
    if method_widget.value != "All":
        subset = subset.loc[
            subset["method_group"].eq(method_widget.value)
        ].copy()

    metric, ascending = RANKING_CRITERIA[criterion_widget.value]
    sort_specification = [(metric, ascending)]
    sort_specification.extend(
        (column, direction)
        for column, direction in TOP_SORT_TIE_BREAKERS
        if column != metric and column in subset.columns
    )
    ordered = subset.sort_values(
        [column for column, _ in sort_specification],
        ascending=[direction for _, direction in sort_specification],
        kind="mergesort",
    ).copy()
    ordered.insert(
        0, "selection_rank", np.arange(1, len(ordered) + 1)
    )
    return ordered.head(top_n_widget.value)

def resolve_viewer_model_path(stored_path):
    try:
        candidate = Path(stored_path)
        if not candidate.is_absolute():
            candidate = REPO_ROOT / candidate
        candidate = candidate.resolve()
    except (OSError, TypeError) as exc:
        raise ValueError(f"Invalid model path: {stored_path!r}") from exc
    try:
        candidate.relative_to(CASE_DIR.resolve())
    except ValueError as exc:
        raise ValueError(
            f"Model path is outside the selected case: {candidate}"
        ) from exc
    if candidate.suffix.lower() != ".pdb" or not candidate.is_file():
        raise FileNotFoundError(f"Model PDB not found: {candidate}")
    return candidate

def safe_path_component(value):
    component = re.sub(
        r"[^A-Za-z0-9._-]+", "_", str(value)
    ).strip("._")
    return component or "selection"

def export_current_top_models(top_models):
    top_models = top_models.copy()
    if top_models.empty:
        raise ValueError("No models are available in the current Top N")

    exported_at = datetime.now(timezone.utc)
    metric, ascending = RANKING_CRITERIA[criterion_widget.value]
    prepared_models = []
    for _, row in top_models.iterrows():
        source_model = resolve_viewer_model_path(row["model_path"])
        method_directory = safe_path_component(row["prediction_method"])
        saved_model = (
            Path("models")
            / method_directory
            / f"{int(row['selection_rank']):03d}_{source_model.name}"
        )
        prepared_models.append((source_model, saved_model))

    saved_paths = [saved_model for _, saved_model in prepared_models]
    if len(saved_paths) != len(set(saved_paths)):
        raise ValueError("The selected models do not have unique export paths")

    timestamp = exported_at.strftime("%Y%m%dT%H%M%S%fZ")
    export_name = "_".join(
        [
            safe_path_component(CASE_ID),
            safe_path_component(metric),
            safe_path_component(method_widget.value),
            f"top{len(top_models)}",
            timestamp,
        ]
    )
    export_directory = SELECTIONS_DIR / export_name
    SELECTIONS_DIR.mkdir(parents=True, exist_ok=True)
    export_directory.mkdir(exist_ok=False)

    try:
        source_model_paths = []
        saved_model_paths = []
        for source_model, saved_model in prepared_models:
            destination = export_directory / saved_model
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_model, destination)
            source_model_paths.append(path_relative_to_repo(source_model))
            saved_model_paths.append(saved_model.as_posix())

        export_table = top_models[TOP_TABLE_COLUMNS].copy()
        export_table["source_model_path"] = source_model_paths
        export_table["saved_model"] = saved_model_paths
        export_table.to_csv(
            export_directory / "selected_ranking.csv",
            index=False,
            encoding="utf-8",
        )

        metadata = {
            "case_id": CASE_ID,
            "exported_at_utc": exported_at.isoformat(),
            "ranking_criterion": criterion_widget.value,
            "ranking_metric": metric,
            "ranking_direction": (
                "ascending" if ascending else "descending"
            ),
            "method_filter": method_widget.value,
            "top_n_requested": int(top_n_widget.value),
            "models_saved": len(export_table),
            "ranked_population": len(zscore_summary),
            "include_af2_unrelaxed": INCLUDE_AF2_UNRELAXED,
            "include_af2_relaxed": INCLUDE_AF2_RELAXED,
            "include_af3_as_predicted": INCLUDE_AF3_AS_PREDICTED,
            "drop_duplicate_af2_finals": DROP_DUPLICATE_AF2_FINALS,
            "self_contained_models": True,
            "table_file": "selected_ranking.csv",
            "models_directory": "models",
        }
        (export_directory / "selection_metadata.json").write_text(
            json.dumps(metadata, indent=2) + "\n",
            encoding="utf-8",
        )
    except Exception:
        shutil.rmtree(export_directory, ignore_errors=True)
        raise

    return export_directory, export_table

def refresh_model_view(*_):
    with viewer_output:
        clear_output(wait=True)
        selection = model_widget.value
        if selection is None or selection not in model_row_lookup:
            print("No model is available for 3D visualization.")
            return
        row = model_row_lookup[selection]
        try:
            model_path = resolve_viewer_model_path(row["model_path"])
            model_text = model_path.read_text(
                encoding="utf-8", errors="replace"
            )
        except (FileNotFoundError, OSError, TypeError, ValueError) as exc:
            print(exc)
            return

        print(
            f"{row['model_id']} | {row['prediction_method']} | "
            f"{model_path.relative_to(REPO_ROOT)}"
        )
        viewer = py3Dmol.view(width=900, height=600)
        viewer.addModel(model_text, "pdb")
        viewer.setStyle({"cartoon": {"color": "spectrum"}})
        viewer.zoomTo()
        display(viewer)

def refresh_top_models(*_):
    top_models = selected_top_models()
    with table_output:
        clear_output(wait=True)
        if top_models.empty:
            print("No models match the selected filters.")
        else:
            display(
                style_zscore_table(top_models[TOP_TABLE_COLUMNS])
            )

    previous_selection = model_widget.value
    model_row_lookup.clear()
    model_options = []
    for position, (_, row) in enumerate(top_models.iterrows(), start=1):
        key = (str(row["method_group"]), str(row["model_id"]))
        model_row_lookup[key] = row
        label = (
            f"{position}. {row['model_id']} "
            f"[{row['prediction_method']}]"
        )
        model_options.append((label, key))

    model_widget.unobserve(refresh_model_view, names="value")
    try:
        model_widget.options = model_options
        available_values = {value for _, value in model_options}
        if previous_selection in available_values:
            model_widget.value = previous_selection
        elif model_options:
            model_widget.value = model_options[0][1]
        else:
            model_widget.value = None
    finally:
        model_widget.observe(refresh_model_view, names="value")
    refresh_model_view()

def save_current_top_n(*_):
    with save_selection_output:
        clear_output(wait=True)
        save_selection_button.disabled = True
        print("Saving the current Top N...")
        try:
            top_models = selected_top_models().copy()
            export_directory, export_table = export_current_top_models(
                top_models
            )
            clear_output(wait=True)
            print(
                f"Saved {len(export_table)} models and the selected table to:"
            )
            print(export_directory.relative_to(REPO_ROOT))
        except Exception as exc:
            clear_output(wait=True)
            print(f"Selection export failed: {exc}")
        finally:
            save_selection_button.disabled = zscore_summary.empty

for control in (criterion_widget, method_widget, top_n_widget):
    control.observe(refresh_top_models, names="value")
model_widget.observe(refresh_model_view, names="value")
save_selection_button.on_click(save_current_top_n)

control_row = widgets.HBox([criterion_widget, method_widget])
save_selection_row = widgets.HBox([save_selection_button])
interactive_panel = widgets.VBox(
    [
        control_row,
        top_n_widget,
        table_output,
        save_selection_row,
        save_selection_output,
        model_widget,
        viewer_output,
    ]
)
_analysis_interactive_observers = [
    (criterion_widget, refresh_top_models, "value"),
    (method_widget, refresh_top_models, "value"),
    (top_n_widget, refresh_top_models, "value"),
    (model_widget, refresh_model_view, "value"),
]
_analysis_interactive_button = (
    save_selection_button,
    save_current_top_n,
)
_analysis_interactive_widgets = [
    criterion_widget,
    method_widget,
    top_n_widget,
    model_widget,
    table_output,
    viewer_output,
    save_selection_output,
    save_selection_button,
    control_row,
    save_selection_row,
    interactive_panel,
]
display(interactive_panel)
refresh_top_models()


## 11. Interpret the main plots

The three outputs provide complementary views of the selected cohort:

1. **Metric distributions** compare the ranges populated by AF2 and AF3.
2. **Z-score density** shows how the normalized confidence, energy and combined metrics are distributed.
3. **Confidence versus pyDock** reveals agreement and disagreement between AlphaFold confidence and pyDock-1VDW energy. The most favourable region combines higher AF-MC with lower pyDock-1VDW.

The figures are shown directly when `DISPLAY_PLOTS=True` and saved beside the CSV files when `GENERATE_PLOTS=True`. These operations do not run AlphaFold, relaxation or pyDock.


In [ ]:
plot_paths = []

if (GENERATE_PLOTS or DISPLAY_PLOTS) and not ranking_export.empty:
    methods = list(ranking_export["prediction_method"].dropna().unique())
    colours = {"AF2": "#377eb8", "AF3": "#e41a1c"}
    metric_labels = {
        "AF-MC": "AF-MC",
        "pyDock-0.1VDW": "pyDock-0.1VDW",
        "pyDock-1VDW": "pyDock-1VDW",
        "Z_AF/pyDock-1VDW": "Z_AF/pyDock-1VDW",
    }

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for axis, column in zip(axes.flatten(), metric_labels):
        for method in methods:
            values = ranking_export.loc[
                ranking_export["prediction_method"].eq(method), column
            ].dropna()
            if not values.empty:
                axis.hist(
                    values,
                    bins=min(25, max(5, int(np.sqrt(len(values))))),
                    alpha=0.55,
                    label=method,
                    color=colours.get(method),
                )
        axis.set_title(metric_labels[column])
        axis.set_ylabel("Models")
        axis.legend()
    fig.tight_layout()
    if GENERATE_PLOTS:
        distribution_path = ANALYSIS_DIR / "metric_distributions.png"
        fig.savefig(distribution_path, dpi=160)
        plot_paths.append(distribution_path)
    if DISPLAY_PLOTS:
        display(fig)
    plt.close(fig)

    fig, axis = plt.subplots(figsize=(7, 4.5))
    zscore_plot_data = [
        ("Z_AF-MC", "Z_AF-MC", "#4daf4a"),
        ("Z_pyDock-0.1VDW", "Z_pyDock-0.1VDW", "#984ea3"),
        ("Z_pyDock-1VDW", "Z_pyDock-1VDW", "#377eb8"),
        ("Z_AF/pyDock-1VDW", "Z_AF/pyDock-1VDW", "#ff7f00"),
    ]
    for column, label, colour in zscore_plot_data:
        values = ranking_export[column].dropna()
        if not values.empty:
            axis.hist(
                values,
                bins=min(30, max(6, int(np.sqrt(len(values))))),
                density=True,
                histtype="step",
                linewidth=2,
                label=label,
                color=colour,
            )
    axis.set_xlabel("Metric value")
    axis.set_ylabel("Density")
    axis.legend()
    fig.tight_layout()
    if GENERATE_PLOTS:
        density_path = ANALYSIS_DIR / "metric_density.png"
        fig.savefig(density_path, dpi=160)
        plot_paths.append(density_path)
    if DISPLAY_PLOTS:
        display(fig)
    plt.close(fig)

    fig, axis = plt.subplots(figsize=(7, 5))
    for method in methods:
        subset = ranking_export.loc[
            ranking_export["prediction_method"].eq(method)
        ]
        axis.scatter(
            subset["pyDock-1VDW"],
            subset["AF-MC"],
            s=24,
            alpha=0.75,
            label=method,
            color=colours.get(method),
        )
    axis.set_xlabel("pyDock-1VDW (lower is better)")
    axis.set_ylabel("AF-MC (higher is better)")
    axis.legend()
    fig.tight_layout()
    if GENERATE_PLOTS:
        scatter_path = ANALYSIS_DIR / "confidence_vs_pydock.png"
        fig.savefig(scatter_path, dpi=160)
        plot_paths.append(scatter_path)
    if DISPLAY_PLOTS:
        display(fig)
    plt.close(fig)

for path in plot_paths:
    print(path)


## 12. Diagnostics

Before interpreting the ranking, use this section to check how many structures entered each stage and why any models were excluded. For `4POU`, the complete reference count is 315 unique AF2 models plus 5 local AF3 models and any available AF3 Server models; a different count is reported but does not by itself stop the notebook.


In [ ]:
inventory_ids = set(
    zip(
        model_inventory.get("case_id", pd.Series(dtype=str)),
        model_inventory.get("prediction_method", pd.Series(dtype=str)),
        model_inventory.get("model_id", pd.Series(dtype=str)),
    )
)
energy_ids = set(
    zip(
        energy_df.get("case_id", pd.Series(dtype=str)),
        energy_df.get("prediction_method", pd.Series(dtype=str)),
        energy_df.get("model_id", pd.Series(dtype=str)),
    )
)
merged_ids = set(
    zip(
        consolidated.get("case_id", pd.Series(dtype=str)),
        consolidated.get("prediction_method", pd.Series(dtype=str)),
        consolidated.get("model_id", pd.Series(dtype=str)),
    )
)

models_missing_energy = inventory_ids - energy_ids
models_missing_confidence = energy_ids - merged_ids
confidence_without_energy = (
    int(af2_merge_counts.get("right_only", 0))
    + int(af3_merge_counts.get("right_only", 0))
)
rmsd_source_counts = energy_df.get(
    "rmsd_source", pd.Series(dtype="string")
).value_counts()

diagnostic_table = pd.DataFrame(
    [
        ("AF2 energy files found", len(AF2_ENERGY_FILES)),
        ("Local AF3 energy files found", len(AF3_LOCAL_ENERGY_FILES)),
        ("AF3 Server energy files found", len(AF3_SERVER_ENERGY_FILES)),
        ("AF2 CAPRI files found", len(AF2_CAPRI_FILES)),
        ("Local AF3 CAPRI files found", len(AF3_LOCAL_CAPRI_FILES)),
        ("AF3 Server CAPRI files found", len(AF3_SERVER_CAPRI_FILES)),
        ("Models using CAPRI metrics", int(rmsd_source_counts.get("capri", 0))),
        (
            "Models using ENE l_RMSD fallback",
            int(rmsd_source_counts.get("ene_fallback", 0)),
        ),
        ("Models without RMSD", int(rmsd_source_counts.get("missing", 0))),
        (
            "CAPRI files with read errors",
            len(energy_diagnostics["capri_read_errors"]),
        ),
        ("AF2 logs found", len(AF2_LOG_FILES)),
        ("Local AF3 confidence JSON found", len(AF3_LOCAL_CONFIDENCE_FILES)),
        ("AF3 Server confidence JSON found", len(AF3_SERVER_CONFIDENCE_FILES)),
        (
            "Energy files with associated PDB",
            len(energy_diagnostics["associated_pdb"]),
        ),
        ("Primary models selected", len(model_inventory)),
        ("Models with energy", len(energy_df)),
        (
            "Models with confidence",
            len(af2_confidence) + len(af3_confidence),
        ),
        ("Models successfully merged", len(consolidated)),
        (
            "Models excluded as duplicate AF2 finals",
            len(energy_diagnostics["excluded_duplicate"]),
        ),
        (
            "Models excluded by cohort flags",
            len(energy_diagnostics["excluded_by_flags"]),
        ),
        ("Energy files missing PDB", len(energy_diagnostics["missing_pdb"])),
        ("Primary models missing energy", len(models_missing_energy)),
        ("Energy models missing confidence", len(models_missing_confidence)),
        ("Confidence records missing energy", confidence_without_energy),
        ("Models excluded for incomplete ranking metrics", int((~rankable).sum())),
        ("Final ranked models", len(ranking_export)),
    ],
    columns=["diagnostic", "count"],
)
print(diagnostic_table.to_string(index=False))

issue_rows = []
for category in (
    "missing_pdb",
    "unrecognized_model",
    "excluded_duplicate",
    "excluded_by_flags",
    "read_errors",
    "capri_read_errors",
):
    for issue in energy_diagnostics[category]:
        issue_rows.append({"category": category, **issue})
for issue in af3_confidence_issues:
    issue_rows.append({"category": "af3_confidence", **issue})

if issue_rows:
    print("\nFile-level diagnostics (first 20):")
    print(pd.DataFrame(issue_rows).head(20).to_string(index=False))
else:
    print("\nNo file-level input issues were detected.")

EXPECTED_COMPLETE_4POU_RANKED_MODELS = (
    320 + len(AF3_SERVER_CONFIDENCE_FILES)
)
if (
    CASE_ID == "4POU"
    and len(ranking_export) != EXPECTED_COMPLETE_4POU_RANKED_MODELS
):
    print(
        "\nNote: the final count differs from the complete, relaxed 4POU "
        f"expectation ({EXPECTED_COMPLETE_4POU_RANKED_MODELS}). "
        "Review the diagnostics above."
    )


## 13. Evidence summary and interpretation template

The tables below assemble the main observations needed for a case-specific assessment: cohort composition, CAPRI-class counts and the models selected by the combined, AlphaFold-confidence and pyDock-energy criteria. They intentionally report evidence without generating a scientific conclusion automatically.

<!--
### Author interpretation

Replace this commented template with the validated interpretation of the selected case.

Suggested points to address:

- Does AlphaFold confidence enrich the best structural-quality classes?
- Do pyDock-0.1VDW, pyDock-1VDW and the combined score select the same models?
- How many independent trajectories, rather than recycle snapshots, support the result?
- Are RMSD, FNC, DockQ and CAPRI mutually consistent, and what uncertainty remains?
- Does the AF-MC threshold motivate additional docking, and how should rescoring be distinguished from generating new docking orientations?
- Is the analysed cohort directly comparable with the cohort described in the chapter?
-->


In [ ]:
case_ranking = ranking_export.loc[
    ranking_export["case_id"].astype("string").eq(str(CASE_ID))
].copy()
case_ranking = case_ranking.sort_values(
    "rank_Z_AF/pyDock-1VDW", kind="mergesort"
)

if case_ranking.empty:
    print(f"No ranked models are available for {CASE_ID}.")
else:
    def readable_method(row):
        method = str(row.get("prediction_method", ""))
        version = str(row.get("af_version", ""))
        if method == "AF2":
            version_match = re.search(r"_v(\d+)$", version)
            if version_match:
                return f"AF2 v{version_match.group(1)}"
        if method == "AF3":
            return "AF3"
        return method or version or "unknown"

    case_ranking["method_label"] = case_ranking.apply(
        readable_method, axis=1
    )

    cohort_table = (
        case_ranking.groupby(
            ["method_label", "relaxation_status"],
            dropna=False,
        )
        .size()
        .rename("models")
        .reset_index()
    )
    print(f"Ranked cohort for {CASE_ID}: {len(case_ranking)} models")
    display(cohort_table.style.hide(axis="index"))

    quality_table = case_ranking[
        ["method_label", "CAPRI"]
    ].copy()
    quality_table["CAPRI"] = quality_table["CAPRI"].fillna(
        "Unavailable"
    )
    quality_table = (
        quality_table.groupby(
            ["method_label", "CAPRI"], dropna=False
        )
        .size()
        .rename("models")
        .reset_index()
    )
    display(quality_table.style.hide(axis="index"))

    selection_specs = [
        ("Best combined score", "Z_AF/pyDock-1VDW", False),
        ("Best AlphaFold confidence", "AF-MC", False),
        ("Best pyDock-1VDW energy", "pyDock-1VDW", True),
        ("Best pyDock-0.1VDW energy", "pyDock-0.1VDW", True),
    ]
    comparison_rows = []
    for criterion, metric, ascending in selection_specs:
        candidates = case_ranking.dropna(subset=[metric]).sort_values(
            [metric, "rank_Z_AF/pyDock-1VDW", "model_id"],
            ascending=[ascending, True, True],
            kind="mergesort",
        )
        if candidates.empty:
            continue
        selected = candidates.iloc[0]
        comparison_rows.append(
            {
                "criterion": criterion,
                "model_id": selected["model_id"],
                "method": selected["method_label"],
                "relaxation_status": selected["relaxation_status"],
                "AF-MC": selected["AF-MC"],
                "pyDock-0.1VDW": selected["pyDock-0.1VDW"],
                "pyDock-1VDW": selected["pyDock-1VDW"],
                "Z_AF/pyDock-1VDW": selected["Z_AF/pyDock-1VDW"],
                "rank_Z_AF/pyDock-1VDW": selected[
                    "rank_Z_AF/pyDock-1VDW"
                ],
                "l_RMSD": selected.get("l_RMSD"),
                "i_RMSD": selected.get("i_RMSD"),
                "FNC": selected.get("FNC"),
                "DockQ": selected.get("DockQ"),
                "CAPRI": selected.get("CAPRI"),
                "rmsd_source": selected.get("rmsd_source"),
            }
        )

    comparison_table = pd.DataFrame(comparison_rows)
    if not comparison_table.empty:
        comparison_table["rank_Z_AF/pyDock-1VDW"] = pd.to_numeric(
            comparison_table["rank_Z_AF/pyDock-1VDW"],
            errors="coerce",
        ).astype("Int64")
        comparison_formatters = {
            column: "{:.3f}"
            for column in (
                "AF-MC",
                "pyDock-0.1VDW",
                "pyDock-1VDW",
                "Z_AF/pyDock-1VDW",
                "l_RMSD",
                "i_RMSD",
                "FNC",
                "DockQ",
            )
            if column in comparison_table.columns
        }
        display(
            comparison_table.style.format(
                comparison_formatters,
                na_rep="",
            ).hide(axis="index")
        )


## 14. Generated files and scope

The notebook writes its main results to `4_Case_Studies/<CASE>/Analysis/`:

- `consolidated_results.csv` contains every successfully matched model and all available metrics.
- `combined_ranking.csv` contains the rankable cohort ordered by `rank_Z_AF/pyDock-1VDW`.
- `metric_distributions.png`, `metric_density.png` and `confidence_vs_pydock.png` are generated when `GENERATE_PLOTS=True`.
- `selections/<selection_name>/` contains the displayed Top N table, selection metadata and self-contained copies of the selected PDB models after using **Save current Top N**.

The confidence–energy ranking remains independent of an experimental structure. When section 3.3.2 has been run with a reference, this notebook reads the associated `.RMSD.capri` table. If the pyDock3 `capriRMSD` step failed but `bindEy` wrote an RMSD in the `.ene` table, that ligand RMSD is retained as a documented fallback while `i_RMSD`, FNC, DockQ and CAPRI remain missing. These annotations do not change the primary ranking.

This notebook analyses existing files; it does not run AlphaFold, structural relaxation or pyDock.

[Back to the notebook guide](#notebook-guide)
